## Q Learning

In [1]:
import random

In [2]:
actions = ["left", "right"] 

In [3]:
Q = {
    "left": 0,
    "right": 0
}

In [4]:
Q

{'left': 0, 'right': 0}

In [5]:
def get_reward(action):
    if action == "right":
        return 1
    else:
        return -1

In [6]:
get_reward("right")

1

In [7]:
get_reward("left")

-1

In [8]:
# learning rate
alpha = 0.1

In [9]:
actions

['left', 'right']

In [11]:
action = random.choice(actions)

In [12]:
action

'right'

In [14]:
Q

{'left': 0, 'right': 0}

In [13]:
Q[action]

0

In [15]:
for i in range(50):
    action = random.choice(actions)
    reward = get_reward(action) 

    Q[action] = Q[action] + alpha * (reward - Q[action])

    print(f"Step {i}, Action: {action}, Reward: {reward}, Q: {Q}")

Step 0, Action: right, Reward: 1, Q: {'left': 0, 'right': 0.1}
Step 1, Action: right, Reward: 1, Q: {'left': 0, 'right': 0.19}
Step 2, Action: left, Reward: -1, Q: {'left': -0.1, 'right': 0.19}
Step 3, Action: right, Reward: 1, Q: {'left': -0.1, 'right': 0.271}
Step 4, Action: left, Reward: -1, Q: {'left': -0.19, 'right': 0.271}
Step 5, Action: left, Reward: -1, Q: {'left': -0.271, 'right': 0.271}
Step 6, Action: left, Reward: -1, Q: {'left': -0.34390000000000004, 'right': 0.271}
Step 7, Action: right, Reward: 1, Q: {'left': -0.34390000000000004, 'right': 0.34390000000000004}
Step 8, Action: right, Reward: 1, Q: {'left': -0.34390000000000004, 'right': 0.40951000000000004}
Step 9, Action: left, Reward: -1, Q: {'left': -0.40951000000000004, 'right': 0.40951000000000004}
Step 10, Action: left, Reward: -1, Q: {'left': -0.46855900000000006, 'right': 0.40951000000000004}
Step 11, Action: right, Reward: 1, Q: {'left': -0.46855900000000006, 'right': 0.46855900000000006}
Step 12, Action: right,

In [16]:
Q

{'left': -0.911370618803475, 'right': 0.94185026299696}

## DQN

In [18]:
import torch
import torch.nn as nn
import torch.optim as optim

In [19]:
class DQN(nn.Module):
    def __init__(self, state_size, action_size):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(state_size, 128), 
            nn.ReLU(), 
            nn.Linear(128, action_size)
        ) 

    def forward(self, x):
        return self.net(x)

In [20]:
state_size = 4
action_size = 2

In [21]:
model = DQN(state_size, action_size)
target_model = DQN(state_size, action_size)

# Copy weights from main model to target model
target_model.load_state_dict(model.state_dict())

optimizer = optim.Adam(model.parameters(), lr=1e-3)
loss_fn = nn.MSELoss()

In [29]:
state = torch.randn(32, state_size)
next_state = torch.randn(32, state_size)
reward = torch.randn(32)
done = torch.randint(0, 2, (32,)).float()
action = torch.randint(0, action_size, (32,))   

def train_step():
    q_values = model(state)
    q_value = q_values.gather(1, action.unsqueeze(1)).squeeze(1)

    with torch.no_grad():
        next_q = target_model(next_state).max(1)[0]
        target = reward + 0.99 * next_q * (1 - done)

    loss = loss_fn(q_value, target)
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    return loss.item()

loss_value = train_step()
print("Loss:", loss_value)

Loss: 0.7676303386688232


In [34]:
import gymnasium as gym
from stable_baselines3 import PPO

In [35]:
env = gym.make("CarRacing-v3", render_mode="human")

In [36]:
model = PPO(
    "CnnPolicy", 
    env,
    verbose=1,
    learning_rate=0.0003,
    n_steps=2048,
    batch_size=64
)

Using cuda device
Wrapping the env with a `Monitor` wrapper
Wrapping the env in a DummyVecEnv.
Wrapping the env in a VecTransposeImage.


In [38]:
model.learn(total_timesteps=100)

---------------------------------
| rollout/           |          |
|    ep_len_mean     | 1e+03    |
|    ep_rew_mean     | -49.9    |
| time/              |          |
|    fps             | 48       |
|    iterations      | 1        |
|    time_elapsed    | 42       |
|    total_timesteps | 2048     |
---------------------------------


In [39]:
obs, _ = env.reset()

In [40]:
for _ in range(1000):
    action, _ = model.predict(obs)
    obs, reward, done, trauncated, info = env.step(action)

    if done or trauncated:
        obs, _ = env.reset()